# ML4VRP 2026 — Colab Bridge
Remote execution environment for the Hierarchical MARL-HGS hybrid solver.

**Runtime:** GPU → T4 (16GB VRAM)

In [1]:
# Cell 1: Clone project from GitHub
import os, sys, shutil

REPO_URL = 'https://github.com/Karam-Kreidli/CVRP.git'
LOCAL_DIR = '/content/ml4vrp'

os.chdir('/content')  # reset CWD before rmtree
if os.path.isdir(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)

!git clone {REPO_URL} {LOCAL_DIR}

if LOCAL_DIR not in sys.path:
    sys.path.insert(0, LOCAL_DIR)
os.chdir(LOCAL_DIR)

# Verify
py_files = sorted(f for f in os.listdir(os.path.join(LOCAL_DIR, 'src')) if f.endswith('.py'))
print(f'\u2713 Cloned {len(py_files)} source files: {py_files}')

Cloning into '/content/ml4vrp'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 54 (delta 20), reused 32 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 1.12 MiB | 11.23 MiB/s, done.
Resolving deltas: 100% (20/20), done.
✓ Cloned 7 source files: ['__init__.py', 'agent_driver.py', 'agent_manager.py', 'main.py', 'model_vision.py', 'solver_engine.py', 'train.py']


In [2]:
# Cell 2: Install Dependencies
import torch
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '')
print(f'PyTorch {TORCH}, CUDA {torch.version.cuda}')

!pip install -q pyvrp gymnasium
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install -q torch-geometric

PyTorch 2.10.0, CUDA 12.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.2/763.2 kB 18.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.0 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 100.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 108.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:0000:01


In [3]:
# Cell 3: Verify imports + run smoke tests
import sys, os

LOCAL_DIR = '/content/ml4vrp'
if LOCAL_DIR not in sys.path:
    sys.path.insert(0, LOCAL_DIR)
os.chdir(LOCAL_DIR)

from src.model_vision import GNNEncoder
from src.agent_manager import FleetManager
from src.agent_driver import RouteDriver
print('\u2713 GNNEncoder, FleetManager, RouteDriver imported OK')

!python -m src.main

✓ GNNEncoder, FleetManager, RouteDriver imported OK
=== Stage 1: Single Instance ===
  Node embeddings: torch.Size([100, 128])
  Graph embedding: torch.Size([1, 128])
  Parameters: 51,584
  PASSED

=== Stage 1: FP16 (400 nodes) ===
  Node embeddings dtype: torch.float32
  Graph embedding dtype:  torch.float32
  PASSED

=== Stage 1: Batched (variable sizes) ===
  Batched node embeddings: torch.Size([1004, 128])
  Batched graph embeddings: torch.Size([4, 128])
  PASSED

=== Stage 2: Fleet Manager ===
  Action logits: torch.Size([4, 3])
  State value:   torch.Size([4, 1])
  Sampled actions: [0, 2, 0, 2]
  Parameters: 12,932
  PASSED

=== Stage 2: Fleet Manager FP16 ===
  Action logits dtype: torch.float16
  State value dtype:   torch.float16
  PASSED

=== Stage 1+2: End-to-End Pipeline ===
  GNN → graph_emb: torch.Size([1, 128])
  Fleet Manager → action: REMOVE, value: -0.0679
  PASSED

=== Stage 3: CVRPEnv (PyVRP wrapper) ===
  Reset → NV=3, TD=318, score=3318
  Step 1:     INTENSIVE_POL

In [4]:
# Cell 4: GPU Smoke Test — 400-node FP16
import torch
from src.model_vision import GNNEncoder

device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

encoder = GNNEncoder().to(device)

N = 401  # depot + 400 customers
x = torch.rand(N, 3, device=device)
x[0, 2] = 0.0
pos = x[:, :2]
batch = torch.zeros(N, dtype=torch.long, device=device)

with torch.amp.autocast('cuda'):
    node_emb, graph_emb = encoder(x, pos, batch)

print(f'Node embeddings: {node_emb.shape}, dtype={node_emb.dtype}')
print(f'Graph embedding: {graph_emb.shape}, dtype={graph_emb.dtype}')
print(f'Parameters: {sum(p.numel() for p in encoder.parameters()):,}')

GPU: Tesla T4
VRAM: 15.6 GB
Node embeddings: torch.Size([401, 128]), dtype=torch.float32
Graph embedding: torch.Size([1, 128]), dtype=torch.float32
Parameters: 51,584


In [5]:
# Cell 5: Memory Profile — Batch=32 x N=400
import torch
from torch_geometric.data import Data, Batch
from src.model_vision import GNNEncoder

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

encoder = GNNEncoder().to(device)

graphs = []
for _ in range(32):
    n = 401
    coords = torch.rand(n, 2)
    demands = torch.rand(n, 1)
    demands[0] = 0.0
    graphs.append(Data(x=torch.cat([coords, demands], dim=-1), pos=coords))

big_batch = Batch.from_data_list(graphs).to(device)

with torch.amp.autocast('cuda'):
    node_emb, graph_emb = encoder(big_batch.x, big_batch.pos, big_batch.batch)

loss = node_emb.sum() + graph_emb.sum()
loss.backward()

peak_mb = torch.cuda.max_memory_allocated() / 1e6
print(f'Peak GPU memory (batch=32, N=400, FP16): {peak_mb:.1f} MB')
print(f'Fraction of T4 VRAM: {peak_mb / 16384 * 100:.1f}%')

Peak GPU memory (batch=32, N=400, FP16): 918.1 MB
Fraction of T4 VRAM: 5.6%


---
## Stage 5: Production Training

In [6]:
# Cell 6: Download Uchoa X-dataset from ML4VRP2026 GitHub repo
import os, glob, shutil

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Check if already downloaded
existing = glob.glob(os.path.join(DATA_DIR, 'X-n*.vrp'))
if len(existing) >= 50:
    print(f'\u2713 {len(existing)} X-instances already in {DATA_DIR}')
else:
    # Clone the competition repo (sparse checkout for instances only)
    !git clone --depth 1 --filter=blob:none --sparse \
        https://github.com/ML4VRP/ML4VRP2026.git /tmp/ml4vrp_repo
    %cd /tmp/ml4vrp_repo
    !git sparse-checkout set Instances/cvrp/vrp
    %cd /content
    # Copy .vrp files to DATA_DIR
    src = '/tmp/ml4vrp_repo/Instances/cvrp/vrp'
    for f in glob.glob(os.path.join(src, 'X-n*.vrp')):
        shutil.copy(f, DATA_DIR)
    shutil.rmtree('/tmp/ml4vrp_repo', ignore_errors=True)

files = sorted(glob.glob(os.path.join(DATA_DIR, 'X-n*.vrp')))
print(f'\u2713 Ready: {len(files)} instances in {DATA_DIR}')
if files:
    print(f'  Range: {os.path.basename(files[0])} ... {os.path.basename(files[-1])}')

Cloning into '/tmp/ml4vrp_repo'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 14 (delta 0), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 8.07 KiB | 8.07 MiB/s, done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 8.49 KiB | 8.49 MiB/s, done.
/tmp/ml4vrp_repo
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 59 (delta 0), reused 59 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 135.74 KiB | 8.48 MiB/s, done.
/content
✓ Ready: 59 instances in /content/data
  Range: X-n101-k25.vrp ... X-n401-k29.vrp


In [7]:
# Cell 7: TrainingDashboard - Live monitoring
import csv, os, time
import matplotlib.pyplot as plt
from IPython.display import clear_output

class TrainingDashboard:
    def __init__(self, csv_path='logs/training_metrics.csv'):
        self.csv_path = csv_path

    def read_metrics(self):
        data = {}
        if not os.path.exists(self.csv_path):
            return data
        with open(self.csv_path, 'r') as f:
            for row in csv.DictReader(f):
                for k, v in row.items():
                    data.setdefault(k, []).append(float(v))
        return data

    def plot(self):
        data = self.read_metrics()
        if not data or 'epoch' not in data:
            print('No training data yet...'); return
        ep = data['epoch']
        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        # NV
        nv = data.get('final_nv', [])
        ax[0].plot(ep, nv, 'b-', lw=1.5)
        ax[0].set_title('Fleet Size (NV)'); ax[0].set_xlabel('Epoch')
        ax[0].grid(True, alpha=0.3)
        if nv: ax[0].axhline(min(nv), color='g', ls='--', alpha=.5, label=f'Best:{min(nv):.0f}'); ax[0].legend()
        # TD
        td = data.get('final_td', [])
        ax[1].plot(ep, td, 'r-', lw=1.5)
        ax[1].set_title('Total Distance (TD)'); ax[1].set_xlabel('Epoch')
        ax[1].grid(True, alpha=0.3)
        if td: ax[1].axhline(min(td), color='g', ls='--', alpha=.5, label=f'Best:{min(td):.0f}'); ax[1].legend()
        # Loss
        ax[2].plot(ep, data.get('mgr_policy_loss',[]), 'b-', lw=1.5, label='Manager')
        ax[2].plot(ep, data.get('drv_policy_loss',[]), 'r-', lw=1.5, label='Driver')
        ax[2].set_title('PPO Policy Loss'); ax[2].set_xlabel('Epoch')
        ax[2].grid(True, alpha=0.3); ax[2].legend()
        plt.tight_layout(); plt.show()
        sc = data.get('final_score',[0])
        print(f'Epochs:{len(ep)} | Best:{min(sc):.0f} | Latest:{sc[-1]:.0f} | NV:{nv[-1]:.0f} | TD:{td[-1]:.0f}')

    def monitor(self, interval=15, max_refreshes=500):
        for _ in range(max_refreshes):
            clear_output(wait=True); self.plot(); time.sleep(interval)

dashboard = TrainingDashboard(os.path.join(LOCAL_DIR, 'logs', 'training_metrics.csv'))
dashboard.plot()

No training data yet...


In [8]:
# Cell 8: Launch Training Run (T4-optimized)
import os, sys
os.chdir(LOCAL_DIR)

!python -m src.main train \
    --instance_path /content/data \
    --epochs 100 \
    --batch_size 16 \
    --manager_lr 1e-4 \
    --driver_lr 5e-4 \
    --episodes_per_epoch 1 \
    --fp16 \
    --ent_coeff 0.05 \
    --log_dir {LOCAL_DIR}/logs \
    --checkpoint_dir {LOCAL_DIR}/checkpoints \
    --save_interval 10 \
    --curriculum_epochs 20

Device: cuda
Instances: 59
Curriculum: N≤100 for first 20 epochs, then N≤400
Training for 100 epochs...
[   1/100] Score:    42911 | NV: 21 | TD:    21911 | MgrPL:  0.0440 | DrvPL: -0.0727 | MgrNrm: 0.000 | DrvNrm: 0.800 | LR: 1.0e-04/5.0e-04
[   2/100] Score:   229241 | NV: 87 | TD:   142241 | MgrPL:  0.1069 | DrvPL: -0.0345 | MgrNrm: 0.464 | DrvNrm: 0.401 | LR: 9.9e-05/4.9e-04
[   3/100] Score:    85416 | NV: 43 | TD:    42416 | MgrPL:  0.0057 | DrvPL: -0.0185 | MgrNrm: 0.334 | DrvNrm: 0.511 | LR: 9.9e-05/4.9e-04
[   4/100] Score:    22785 | NV: 10 | TD:    12785 | MgrPL: -0.0578 | DrvPL: -0.0252 | MgrNrm: 0.152 | DrvNrm: 0.350 | LR: 9.8e-05/4.9e-04
[   5/100] Score:    66868 | NV: 28 | TD:    38868 | MgrPL:  0.0129 | DrvPL:  0.0438 | MgrNrm: 0.135 | DrvNrm: 0.893 | LR: 9.8e-05/4.9e-04
[   6/100] Score:    47051 | NV: 18 | TD:    29051 | MgrPL: -0.1622 | DrvPL: -0.0907 | MgrNrm: 0.123 | DrvNrm: 0.598 | LR: 9.7e-05/4.9e-04
[   7/100] Score:    80957 | NV: 29 | TD:    51957 | MgrPL: -0

In [ ]:
# Cell 9: Live Dashboard (run after launching training)
dashboard.monitor(interval=15)

In [ ]:
# Cell 10: Download trained model
import os
from google.colab import files

model_path = os.path.join(LOCAL_DIR, 'logs', 'best_model.pth')
if os.path.exists(model_path):
    files.download(model_path)
    print(f'✓ Downloading {model_path}')
else:
    print(f'✗ No model found at {model_path} — has training completed at least one epoch?')